In [1]:
import loaders
import pandas as pd
import numpy as np

In [2]:
bible = loaders.loader('bible')
references = loaders.loader('clustered_references')
references['ot_loc'] = bible.iloc[references['ostart']].index

In [ ]:
rev_refs = references[bible.iloc[references['nstart']]['book'].values == 'Revelation']
bible['referencer'] = False
bible['referencer_0'] = False
bible['referencer_1'] = False
bible['referencer_2'] = False
for start, end, cluster in rev_refs[['nstart', 'nend', 'cluster']].values:
    bible.loc[bible.index[start:end], 'referencer'] = True
    bible.loc[bible.index[start:end], f'referencer_{cluster}'] = True
rev = bible.loc['Revelation']

In [ ]:
top_10 = bible['str'].value_counts()[:10].index

In [ ]:
for word in top_10:
    rmac = bible.query('str == @word').iloc[0]['rmac']
    print(word, round((bible['str'] == word).sum() * 100/ len(bible), 2), rmac.split('-')[0])

In [ ]:
bible['str'].isin(top_10).sum() / len(bible) * 100

In [ ]:
# distribution of reference lengths
rev_refs['nlen'].value_counts().plot.bar()

In [ ]:
pd.DataFrame({
    'Revelation':rev_refs.query('nlen > 5')['ot_loc'].str[0].value_counts()/len(rev_refs),
    'New Testament':references.query('(nlen > 5)')['ot_loc'].str[0].value_counts()/len(references)
}).sort_values('Revelation', ascending=False).plot.barh();

In [ ]:
# proportion of words in revelation that are referencers
rev['referencer'].value_counts()

In [ ]:
import matplotlib.pyplot as plt

In [ ]:
fig, axes = plt.subplots(nrows=3)
for label in range(3):
    cluster = bible.query(f'referencer_{label}')
    per_chapter_props = cluster.groupby(cluster['chapter'])[f'referencer_{label}'].apply(lambda g: sum(g)/len(g)).sort_index()
    axes[label].scatter(per_chapter_props.index, per_chapter_props.values)
    axes[label].title(f'Per-chapter proportion of referencing words in the book of Revelation (Cluster {label})')